# B2.7 · Failure taxonomy

**Function B — Product & Application Security → The Security Automation / Harness Engineer**  ·  *AI for Security*

---

**Risk.** Loop divergence, objective drift, reward hacking, silent truncation, tool thrash.

**Control.** Recognise each from a trace.

**This lab.** Name the failure from the trace alone.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("B2.7"))

A failure taxonomy turns 'the agent messed up' into something you can count, route and fix. Without one, every incident is novel.

In [ ]:
TAXONOMY = {
 "capability":   ("the model could not do it",        "better model, better context"),
 "verification": ("it did it wrong and we believed it", "fix the verifier — B2.2"),
 "authority":    ("it did something it should not be able to do", "fix scope — A2"),
 "containment":  ("the action reached further than intended",     "fix the sandbox — A3"),
 "injection":    ("it was told to by untrusted content",          "provenance — M0.4"),
 "budget":       ("it never stopped",                             "stop conditions — B2.4"),
 "idempotency":  ("it did the right thing twice",                 "replay keys — B2.9"),
}
for k, (what, fix) in TAXONOMY.items():
    print(f"{k:14s} {what:44s} → {fix}")

Classify a real run. The point of the taxonomy is that the *fix owner* differs per class — verification failures go to the harness engineer, authority failures go to identity, and confusing the two wastes a quarter.

In [ ]:
from cybercommons import loop

BROKEN = "def add(a, b): return a - b"
tr = loop.run(loop.FakeModel([BROKEN]), loop.llm_judge(), max_steps=3)
print(tr.table())
print("\nclassification: VERIFICATION failure.")
print("The model produced wrong code (capability) but the harness *shipped* it,")
print("and that is a different defect with a different owner.")

### Expect

The taxonomy prints with a fix owner per class, and the sample run is classified as a verification failure rather than a capability one.

### Your turn

Take your last five agent incidents and assign exactly one class to each. Any incident needing two classes is really two incidents.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/B2.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*